In [1]:
# # Cloning GitHub repo
# !git clone https://github.com/Romit-M/UIDAI-Hackathon-2026.git
# %cd UIDAI-Hackathon-2026

# # IMPORTS
# !pip install rapidfuzz -q
from rapidfuzz import process, utils

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import sys
from pathlib import Path
import os
# HELPER FUNCTIONS

# Getting parent directory
PARENT_DIR = str(Path.cwd().parent)
sys.path.insert(0, PARENT_DIR)
print("Added to sys.path:", PARENT_DIR)


# Data loader
def load_data(file_path):
    df = pd.read_csv(file_path)
    df.columns = df.columns.str.lower()
    return df


Added to sys.path: /home/kamal/Desktop/UIDAI-Hackathon-2026


### **DATA CLEANING**

In [2]:
# TEXT NORMALIZATION (making text comparable)


def normalize_text(s):
    return s.lower().strip().replace("&", "and").replace(" ", "")


In [3]:
# UNIQUE STATE MAPPING USING FUZZY LOGIC

# Manual override for historic name changes
manual_fixes = {
    "orissa": "odisha",
    "pondicherry": "puducherry",
    "uttaranchal": "uttarakhand",
    "damananddiu": "dadraandnagarhavelianddamananddiu",
    "dadraandnagarhaveli": "dadraandnagarhavelianddamananddiu",
}

# Official Aadhaar/UIDAI standard list
official_states = json.load(open(f"{PARENT_DIR}/data/states.json", "r"))[
    "official_states"
]


def get_closest_match(key: str, ground: list[str], score_cutoff: int = 80) -> str:
    """Give me the doubttful key and I will return the closest match from the grounding list using fuzzy matching.

    Args:
        key (str): the value to be matched
        ground (list[str]): the list of possible matches
        score_cutoff (int): minimum score to consider a match. Defaults to 80.

    Returns:
        str: the closest match if found, else the original key
    """
    # Returns the best match from official_states if score > 80%
    match = process.extractOne(key, ground, score_cutoff=score_cutoff)
    return match[0] if match else key


def state_mapping(df):
    # Manual override
    df["state"] = df["state"].replace(manual_fixes)

    # Get unique dirty states
    dirty_states = df["state"].unique()

    # Create and apply an automated mapping dictionary
    auto_mapping = {ds: get_closest_match(ds, official_states) for ds in dirty_states}
    df["state"] = df["state"].map(auto_mapping)

    return df


In [4]:
# CLEANING PIPELINE


def clean_pipeline(df):
    df["state"] = df["state"].astype(str).apply(normalize_text)
    df["district"] = df["district"].astype(str).apply(normalize_text)

    df = state_mapping(df)

    return df


In [7]:
# DATA LOADING -> CLEANING -> SAVING CLEANED DATA


DATA_PATH = f"{PARENT_DIR}/data"
RAW_PATH = f"{DATA_PATH}/raw"
PROCESSED_PATH = f"{DATA_PATH}/processed"

for category in os.listdir(RAW_PATH):
    # only for folders
    if os.path.isdir(f"{RAW_PATH}/{category}"):
        for r in os.listdir(f"{RAW_PATH}/{category}"):
            filename = f"{RAW_PATH}/{category}/{r}"
            # Load file
            df = load_data(filename)

            # Apply cleaning
            df = clean_pipeline(df)

            # Save cleaned file
            output_filename = f"{r[:-4]}_cleaned.csv"
            output_path = f"{PROCESSED_PATH}/{category}/{output_filename}"
            df.to_csv(output_path, index=False)

            print(f"{output_filename} saved.")


api_data_aadhar_biometric_0_500000_cleaned.csv saved.
api_data_aadhar_biometric_500000_1000000_cleaned.csv saved.
api_data_aadhar_biometric_1500000_1861108_cleaned.csv saved.
api_data_aadhar_biometric_1000000_1500000_cleaned.csv saved.
api_data_aadhar_demographic_0_500000_cleaned.csv saved.
api_data_aadhar_demographic_2000000_2071700_cleaned.csv saved.
api_data_aadhar_demographic_1500000_2000000_cleaned.csv saved.
api_data_aadhar_demographic_1000000_1500000_cleaned.csv saved.
api_data_aadhar_demographic_500000_1000000_cleaned.csv saved.
api_data_aadhar_enrolment_1000000_1006029_cleaned.csv saved.
api_data_aadhar_enrolment_0_500000_cleaned.csv saved.
api_data_aadhar_enrolment_500000_1000000_cleaned.csv saved.


In [ ]:
# PUSH CLEANED FILES TO REPO

from google.colab import userdata

pat = userdata.get("GitHubAccessToken")

!git config --global user.name "Romit-M"
!git config --global user.email "romitrmaity@gmail.com"

!git add .
!git commit -m "Added cleaned data files"
!git push https://{pat}@github.com/Romit-M/UIDAI-Hackathon-2026.git


In [ ]:
# !rm -rf /content/UIDAI-Hackathon-2026
# %cd /content
